# Semana 1: carga y reporte de Fast_Food_Restaurants

En este notebook se construye un flujo completo de trabajo:

1. Configurar la ruta del proyecto para importar las clases reutilizables.
2. Cargar `Fast_Food_Restaurants` desde Hugging Face.
3. Obtener un `DataFrame` de pandas y revisar su estructura.
4. Generar un reporte en pantalla y guardarlo en un archivo `.txt`.

Ejecuta las celdas en orden. Si el dataset requiere autenticación, inicia sesión previamente con `hf auth login`.

## 1. Preparar el entorno e importar la clase de carga

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`. Así se puede importar `HFDatasetLoader` sin instalar el proyecto como paquete.

In [5]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

## 2. Configurar el dataset y la salida

`dataset_name` identifica el repositorio en Hugging Face y `split="train"` indica la división que se descargará. El parámetro `filename` hace que el `DataFrame` también se guarde como CSV en `src/sincronicas/datos/` cuando el notebook se ejecuta desde esa carpeta.

In [6]:
loader = HFDatasetLoader(
    dataset_name="Bizwhiznetwork/pizza_sales",
    split="train",
    filename="datos/pizza_sales.csv",
)


## 3. Descargar el dataset y crear el DataFrame

El método `load()` descarga el dataset, lo convierte a `pandas.DataFrame` y lo asigna a la variable `df`. La expresión `df.head()` muestra las primeras filas para comprobar que la carga fue exitosa.

In [7]:
df = loader.load()
df.head()

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1.0,1.0,hawaiian_m,1.0,1/1/2015,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2.0,2.0,classic_dlx_m,1.0,1/1/2015,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3.0,2.0,five_cheese_l,1.0,1/1/2015,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4.0,2.0,ital_supr_l,1.0,1/1/2015,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5.0,2.0,mexicana_m,1.0,1/1/2015,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


## 4. Revisar la estructura del DataFrame

Esta celda realiza una revisión inicial: cuenta filas y columnas y muestra el tipo de cada columna junto con información de memoria y valores no nulos. Esta inspección ayuda a detectar problemas antes de analizar los datos.

In [8]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 48620
Columnas: 12
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pizza_id           48620 non-null  float64
 1   order_id           48620 non-null  float64
 2   pizza_name_id      48620 non-null  object 
 3   quantity           48620 non-null  float64
 4   order_date         48620 non-null  object 
 5   order_time         48620 non-null  object 
 6   unit_price         48620 non-null  float64
 7   total_price        48620 non-null  float64
 8   pizza_size         48620 non-null  object 
 9   pizza_category     48620 non-null  object 
 10  pizza_ingredients  48620 non-null  object 
 11  pizza_name         48620 non-null  object 
dtypes: float64(5), object(7)
memory usage: 4.5+ MB


## 1.1 Preparar el entorno e importar las clases

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`. Así se pueden importar las clases sin instalar el proyecto como paquete.

In [9]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader, Reportes
from clases_po import PreprocesadorPizzas, LLMNvidia, EnriquecedorPizzeria
from clases_po import AnalisisBase, GraficosBase, AnalisisPizzeria, GraficosPizzeria

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1.2 Configurar el dataset y la salida

`dataset_name` identifica el repositorio en Hugging Face y `split="train"` indica la división que se descargará. El parámetro `filename` hace que el `DataFrame` también se guarde como CSV en `src/sincronicas/datos/` cuando el notebook se ejecuta desde esa carpeta.

In [14]:
loader = HFDatasetLoader(
    dataset_name="Bizwhiznetwork/pizza_sales",
    split="train",
    filename="datos/pizza_sales.csv",
)

---
# PARTE 2 · Preprocesamiento y estandarización (clase `PreprocesadorPizzas`)

El Excel de clase trabaja con las **primeras 1.700 líneas** (1 al 13 de enero de 2015; el día 13 está incompleto). `filas_clase=1700` toma exactamente ese corte para que los resultados sean comparables. Con `filas_clase=None` se usaría todo el dataset (todo 2015).

## 2.1 Diagnóstico de tipos (hoja **Tipos**, sección A)

Equivale a `ESTEXTO`, `ESNUMERO`, `CONTARA` y `CONTAR` por columna.

In [15]:
pre = PreprocesadorPizzas(filas_clase=1700)

df_base = pre.recortar(df)
pre.diagnosticar(df_base)

,columna,tipo_actual,ejemplo,no_vacíos (CONTARA),convertibles_a_número,¿es número? (ESNUMERO)
0,pizza_id,float64,1.0,1700,1700,True
1,order_id,float64,1.0,1700,1700,True
2,pizza_name_id,object,hawaiian_m,1700,0,False
3,quantity,float64,1.0,1700,1700,True
4,order_date,object,1/1/2015,1700,0,False
5,order_time,object,11:38:36,1700,0,False
6,unit_price,float64,13.25,1700,1700,True
7,total_price,float64,13.25,1700,1700,True
8,pizza_size,object,M,1700,0,False
9,pizza_category,object,Classic,1700,0,False


**Lectura:** fecha y hora llegan como **texto** (y la fecha mezcla `1/1/2015` con `13-01-2015`), los identificadores como decimales (`1.0`), y tamaño y categoría como texto libre.

## 2.2 Diccionario de columnas (hoja **Tipos**, sección B)

In [16]:
pre.diccionario()

,columna,qué significa,tipo de variable,tipo en pandas
0,pizza_id,Número de la línea,Cualitativa (identificador),Int64 (entero)
1,order_id,Número del pedido,Cualitativa (identificador),Int64 (entero)
2,pizza_name_id,Código de pizza y tamaño,Cualitativa nominal,string
3,quantity,Pizzas en la línea,Cuantitativa discreta,int64
4,order_date,Fecha del pedido (día/mes/año),Cuantitativa (tiempo),datetime64
5,order_time,Hora del pedido,Cuantitativa (tiempo),time (hh:mm:ss)
6,unit_price,Precio de una pizza,Cuantitativa continua,"float64 (moneda, 2 dec.)"
7,total_price,Cantidad × precio unitario,Cuantitativa continua,"float64 (moneda, 2 dec.)"
8,pizza_size,Tamaño,Cualitativa ordinal (S < M < L < XL),category ordenada
9,pizza_category,Categoría,Cualitativa nominal,category


## 2.3 Convertir tipos y agregar columnas calculadas (hojas **Tipos** D-E y **Datos**)

| Excel | Método de la clase |
|---|---|
| `VALOR.NUMERO`, `SUSTITUIR` + `FECHA`, `HORANUMERO`, formato Texto | `convertir_tipos()` |
| `total_calculado`, `verificación`, `día_semana`, `hora`, `nuevo_pedido` | `agregar_columnas_calculadas()` |
| Los dos pasos seguidos | `procesar()` |

In [18]:
datos = pre.procesar(df)
datos.head()

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name,fecha_hora,total_calculado,verificacion,dia_semana,hora,nuevo_pedido
0,1,1,hawaiian_m,1,2015-01-01,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza,2015-01-01 11:38:36,13.25,OK,jueves,11,1
1,2,2,classic_dlx_m,1,2015-01-01,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza,2015-01-01 11:57:40,16.00,OK,jueves,11,1
2,3,2,five_cheese_l,1,2015-01-01,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza,2015-01-01 11:57:40,18.50,OK,jueves,11,0
3,4,2,ital_supr_l,1,2015-01-01,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza,2015-01-01 11:57:40,20.75,OK,jueves,11,0
4,5,2,mexicana_m,1,2015-01-01,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza,2015-01-01 11:57:40,16.00,OK,jueves,11,0


In [19]:
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1700 entries, 0 to 1699
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   pizza_id           1700 non-null   Int64         
 1   order_id           1700 non-null   Int64         
 2   pizza_name_id      1700 non-null   string        
 3   quantity           1700 non-null   int64         
 4   order_date         1700 non-null   datetime64[ns]
 5   order_time         1700 non-null   object        
 6   unit_price         1700 non-null   float64       
 7   total_price        1700 non-null   float64       
 8   pizza_size         1700 non-null   category      
 9   pizza_category     1700 non-null   category      
 10  pizza_ingredients  1700 non-null   string        
 11  pizza_name         1700 non-null   string        
 12  fecha_hora         1700 non-null   datetime64[ns]
 13  total_calculado    1700 non-null   float64       
 14  verifica

## 2.4 Validaciones de calidad

In [20]:
pre.validar(datos)

✔ Sin valores nulos tras convertir 
✔ Sin filas duplicadas 
✔ total_price = quantity × unit_price (0 filas a revisar)
✔ Fechas en orden cronológico según order_id 

Comparación con el Excel de clase:
✔ Ventas totales   = $28,663.15 → $28,663.15
✔ Pizzas vendidas  = 1,734 → 1,734
✔ Líneas de venta  = 1,700 → 1,700
✔ Pedidos          = 755 → 755

Rango de fechas: 01/01/2015 a 13/01/2015 (13 días; el último tiene 50 líneas)


True

In [22]:
datos.to_csv("datos/pizza_base_procesada.csv", index=False)
reportes.guardar_reporte(datos, "reportes/pizza_base_procesada.txt")
reportes.imprimir_reporte(datos)

REPORTE DEL DATAFRAME
Filas: 1700
Columnas: 18
Duplicados: 0

Columnas y tipos:
- pizza_id: Int64
- order_id: Int64
- pizza_name_id: string
- quantity: int64
- order_date: datetime64[ns]
- order_time: object
- unit_price: float64
- total_price: float64
- pizza_size: category
- pizza_category: category
- pizza_ingredients: string
- pizza_name: string
- fecha_hora: datetime64[ns]
- total_calculado: float64
- verificacion: object
- dia_semana: category
- hora: int32
- nuevo_pedido: int64

Valores nulos por columna:
- pizza_id: 0
- order_id: 0
- pizza_name_id: 0
- quantity: 0
- order_date: 0
- order_time: 0
- unit_price: 0
- total_price: 0
- pizza_size: 0
- pizza_category: 0
- pizza_ingredients: 0
- pizza_name: 0
- fecha_hora: 0
- total_calculado: 0
- verificacion: 0
- dia_semana: 0
- hora: 0
- nuevo_pedido: 0

Estadísticas numéricas:
       pizza_id  order_id  quantity  unit_price  total_price  total_calculado      hora  nuevo_pedido
count    1700.0    1700.0  1700.000    1700.000     170

---
# PARTE 3 · Análisis del Excel con los datos originales (sin enriquecer)

Antes de enriquecer, se replica **todo el análisis del libro de Excel** sobre la base tal como viene: **una sola pizzería, 1 al 13 de enero de 2015, 1.700 líneas**. Así los resultados se pueden comparar uno a uno con las hojas del Excel.

- `AB = AnalisisBase(datos, dias_incompletos=["2015-01-13"])` calcula las tablas. El 13/01 está incompleto (50 líneas): cuenta en los totales, pero se excluye de los promedios por día de la semana para no castigar al martes.
- `GB = GraficosBase(AB)` dibuja los 9 gráficos del Excel y el dashboard.

In [23]:
AB = AnalisisBase(datos, pizza_elegida="The Pepperoni Pizza", dias_incompletos=["2015-01-13"])
GB = GraficosBase(AB)
print("Periodo analizado:", AB.periodo())

Periodo analizado: 01/01/2015 a 13/01/2015 (13 días)


## 3.1 Fórmulas básicas (hoja **Fórmulas**)

Cada fórmula del Excel con su equivalente en pandas y el resultado: operaciones con la fila 185 (3 Spicy Italian), funciones de resumen, valores únicos y una condición.

In [24]:
AB.formulas_basicas(fila_excel=185)

,#,qué quiero saber,fórmula en Excel,equivalente en pandas,resultado
0,1,Multiplicar: total de la línea,=Datos!D185*Datos!G185,V.quantity[i] * V.unit_price[i],62.25
1,2,Dividir: precio de cada pizza,=Datos!H185/Datos!D185,V.total_price[i] / V.quantity[i],20.75
2,3,Sumar: pizzas en dos líneas,=Datos!D185+Datos!D186,V.quantity[i] + V.quantity[i+1],4.0
3,4,Restar: precio más alto − más bajo,=MAX(G)-MIN(G),V.unit_price.max() - V.unit_price.min(),15.75
4,5,Redondear el precio promedio,=REDONDEAR(PROMEDIO(G);2),"round(V.unit_price.mean(), 2)",16.53
5,6,Ventas totales,=SUMA(H),V.total_price.sum(),28663.15
6,7,Pizzas vendidas,=SUMA(D),V.quantity.sum(),1734.0
7,8,Líneas con número,=CONTAR(H),V.total_price.count(),1700.0
8,9,Líneas con nombre de pizza,=CONTARA(L),V.pizza_name.notna().sum(),1700.0
9,10,Celdas vacías en la categoría,=CONTAR.BLANCO(J),V.pizza_category.isna().sum(),0.0


## 3.2 Resumen del negocio (hoja **Descriptiva**, sección A)

In [25]:
## 3.2 Resumen del negocio (hoja **Descriptiva**, sección A)

In [26]:
display(AB.estilo(AB.resumen_total()))
print("Ojo: el último día (13/01) está incompleto, así que la venta promedio por día subestima un poco.")

,Empresa
Ventas totales,"28,663.15"
Pizzas vendidas,"1,734.00"
Líneas de pedido,"1,700.00"
Pedidos,755.00
Ticket promedio,37.96
Pizzas por pedido,2.30
Precio promedio por pizza,16.53
Días con ventas,13.00
Venta promedio por día,"2,204.86"


Ojo: el último día (13/01) está incompleto, así que la venta promedio por día subestima un poco.
